In [1]:
from neuromeka import IndyDCP3 as RobotClient

In [2]:
robot = RobotClient(robot_ip="192.168.0.180")

In [47]:
robot.get_robot_data()["op_state"]

5

In [46]:
robot.recover()

{'code': '0', 'msg': ''}

In [12]:
robot.get_robot_data()["p"]

[-0.012781437,
 9.22838e-05,
 633.00665,
 2.586361e-23,
 -0.00038001713,
 3.3992353e-06,
 351.30765,
 451.25522,
 370.2955,
 20.074049,
 -178.6261,
 69.51132,
 353.71042,
 -450.2164,
 373.17413,
 158.85532,
 -1.7614689,
 110.06253]

In [11]:
len(robot.get_robot_data()["pdot"])

18

In [38]:
import time

# Commands used by the PSF nrmk_motion DH-gripper backend.
GRIPPER_INIT = 0
GRIPPER_SET_POSITION = 2
DH_GRIPPER = 2

def gripper_activate(tool_index=0):
    """Initialize/activate the DH gripper."""
    return robot.set_gripper_command(
        command=GRIPPER_INIT,
        gripper_type=DH_GRIPPER,
        pvt_data=[0, 0, 0, 0],
        tool_index=tool_index,
    )

def gripper_move(position, tool_index=0):
    """Set DH gripper position (0=closed, 1000=open)."""
    position = max(0, min(1000, int(position)))
    return robot.set_gripper_command(
        command=GRIPPER_SET_POSITION,
        gripper_type=DH_GRIPPER,
        pvt_data=[position, 0, 0, 0],
        tool_index=tool_index,
    )

def gripper_open(tool_index=0):
    return gripper_move(position=1000, tool_index=tool_index)

def gripper_close(tool_index=0):
    return gripper_move(position=0, tool_index=tool_index)

# Initialize once before sending open/close commands.
# gripper_activate()
# time.sleep(2.0)
gripper_open()
# gripper_close(tool_index=0)

{}

In [4]:
robot.get_compliance_mode()

{'stiffness': [100,
  100,
  100,
  100,
  100,
  100,
  100,
  100,
  100,
  100,
  100,
  100,
  100,
  100,
  100,
  100,
  100,
  100,
  100,
  100,
  100,
  100],
 'enable': False}

In [19]:
robot.set_compliance_mode(enable=True, stiffness=[100]*22)

{'msg': 'SetSensorlessComplianceMode Success', 'code': '0'}

In [15]:
NOMINAL_HOME_POS = robot.get_robot_data()["q"]

In [16]:
import numpy as np

TARGET_POS = np.array(NOMINAL_HOME_POS)
TARGET_POS[:18] += 10

In [43]:
robot.set_compliance_mode(enable=True, stiffness=[100]*22)

VEL_RATIO = 10
ACC_RATIO = 10

robot.movej(jtarget=NOMINAL_HOME_POS, vel_ratio=VEL_RATIO, acc_ratio=ACC_RATIO)

{'code': '0', 'msg': ''}

In [41]:
from neuromeka import StopCategory
robot.stop_motion(stop_category=StopCategory.CAT2)

{'msg': 'StopMotion Success', 'code': '0'}

In [ ]:
# Open the gripper
gripper_open()

_InactiveRpcError: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.CANCELLED
	details = "Gripper read did not complete"
	debug_error_string = "UNKNOWN:Error received from peer  {grpc_message:"Gripper read did not complete", grpc_status:1, created_time:"2026-08-13T17:44:04.99821078+09:00"}"
>

In [ ]:
# Close the gripper
gripper_close()

In [4]:
# FK/IK test using the robot's current state. This cell does not move the robot.
import numpy as np

ARM_INDEX = 1  # 0: head, 1: left arm, 2: right arm
ACTIVE_JOINT_DOF = 18
ROBOT_JOINT_DOF = 22
TASK_DOF = 6

state = robot.get_robot_data()
current_q = np.asarray(state["q"], dtype=np.float64)
task_start = ARM_INDEX * TASK_DOF
current_task = np.asarray(
    state["p"][task_start:task_start + TASK_DOF],
    dtype=np.float64,
)

assert current_q.shape == (ROBOT_JOINT_DOF,), current_q.shape
assert current_task.shape == (TASK_DOF,), current_task.shape

fk_result = robot.forward_kin(
    jpos=current_q.tolist(),
    arm_index=ARM_INDEX,
)
ik_result = robot.inverse_kin(
    tpos=current_task.tolist(),
    init_jpos=current_q.tolist(),
    arm_index=ARM_INDEX,
)

print("current q22:", current_q.tolist())
print("current task pose:", current_task.tolist())
print("FK response:", fk_result.get("response"))
print("IK response:", ik_result.get("response"))

if str(fk_result.get("response", {}).get("code")) == "0":
    fk_task = np.asarray(fk_result["tpos"], dtype=np.float64)
    assert fk_task.shape == (TASK_DOF,), fk_task.shape
    print("FK task pose:", fk_task.tolist())
    print("FK - current task:", (fk_task - current_task).tolist())
else:
    print("FK failed:", fk_result)

if str(ik_result.get("response", {}).get("code")) == "0":
    ik_q = np.asarray(ik_result["jpos"], dtype=np.float64)
    if ik_q.shape == (ACTIVE_JOINT_DOF,):
        ik_q22 = np.concatenate([ik_q, np.zeros(4, dtype=np.float64)])
    elif ik_q.shape == (ROBOT_JOINT_DOF,):
        ik_q22 = ik_q.copy()
    else:
        raise ValueError(f"Unexpected IK joint shape: {ik_q.shape}")
    print("IK active joints:", ik_q.tolist())
    print("IK q22 with dummy zeros:", ik_q22.tolist())
    print("IK active - current active q:",
          (ik_q22[:ACTIVE_JOINT_DOF] - current_q[:ACTIVE_JOINT_DOF]).tolist())
else:
    print("IK failed:", ik_result)

current q22: [-0.01835927, 0.019959455, 0.030141018, 0.019280462, -138.69981, -25.197933, 76.68723, -99.8471, 77.274055, -37.65436, 4.343033, 137.05882, 25.108269, -73.13261, 102.04383, -73.95116, 37.882317, 0.01665802, 0.0, 0.0, 0.0, 0.0]
current task pose: [351.49078, 466.72107, 370.1288, 20.043913, -178.61748, 69.50214]
FK response: {'msg': 'ForwardKinematics Success', 'code': '0'}
IK response: {'msg': 'InverseKinematics Success', 'code': '0'}
FK task pose: [351.49078, 466.72107, 370.12878, 20.043911, -178.61748, 69.50214]
FK - current task: [0.0, 0.0, -2.0000000006348273e-05, -1.999999998503199e-06, 0.0, 0.0]
IK active joints: [-0.01835927, 0.019959455, 0.030141018, 0.019280462, -138.69981, -25.197933, 76.68723, -99.8471, 77.274055, -37.65436, 4.343033, 137.05882, 25.108269, -73.13261, 102.04383, -73.95116, 37.882317, 0.01665802]
IK q22 with dummy zeros: [-0.01835927, 0.019959455, 0.030141018, 0.019280462, -138.69981, -25.197933, 76.68723, -99.8471, 77.274055, -37.65436, 4.343033, 